In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
import tensorflow as tf
import tensorflow_decision_forests as tfdf

In [51]:
train_df = pd.read_csv("/Users/aleksandr/Документы/TitanicML/data/train.csv")
test_df = pd.read_csv("/Users/aleksandr/Документы/TitanicML/data/test.csv")

train_df.head(10)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


In [52]:
def preprocess(df):
    df = df.copy()
    def normaloize_name(x):
        return " ".join([v.strip(",()[].\"'") for v in x.split(" ")])
    def ticket_number(x):
        return x.split(" ")[-1]
    def ticket_item(x):
        items = x.split(" ")
        if len(items) == 1:
            return "None"
        return "_".join(items[0:-1])
    
    df["Name"] = df["Name"].apply(normaloize_name)
    df["Ticket_number"] = df["Ticket"].apply(ticket_number)
    df["Ticket_item"] = df["Ticket"].apply(ticket_item)
    return df

preprocess_train_df = preprocess(train_df)
preprocess_test_df = preprocess(test_df)

categorical_columns = preprocess_train_df.select_dtypes(
    include=["object", "category"]
).columns

for col in categorical_columns:
    preprocess_train_df[col] = (
        preprocess_train_df[col]
        .fillna("__MISSING__")
        .astype(str)
    )

    preprocess_test_df[col] = (
        preprocess_test_df[col]
        .fillna("__MISSING__")
        .astype(str)
    )

preprocess_train_df.head()

/var/folders/3z/c5bxtqgx0yj70y6vbqnfwxtw0000gn/T/ipykernel_54728/632754599.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = preprocess_train_df.select_dtypes(


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Ticket_number,Ticket_item
0,1,0,3,Braund Mr Owen Harris,male,22.0,1,0,A/5 21171,7.2500,__MISSING__,S,21171,A/5
1,2,1,1,Cumings Mrs John Bradley Florence Briggs Thayer,female,38.0,1,0,PC 17599,71.2833,C85,C,17599,PC
2,3,1,3,Heikkinen Miss Laina,female,26.0,0,0,STON/O2. 3101282,7.9250,__MISSING__,S,3101282,STON/O2.
3,4,1,1,Futrelle Mrs Jacques Heath Lily May Peel,female,35.0,1,0,113803,53.1000,C123,S,113803,None
4,5,0,3,Allen Mr William Henry,male,35.0,0,0,373450,8.0500,__MISSING__,S,373450,None


In [53]:
input_features = list(preprocess_train_df.columns)
input_features.remove("Ticket")
input_features.remove("PassengerId")
input_features.remove("Survived")

print(f"Input features: {input_features}")

Input features: ['Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Cabin', 'Embarked', 'Ticket_number', 'Ticket_item']


In [54]:
def tokenize_names(features, labels=None):
    features["Name"] =  tf.strings.split(features["Name"])
    return features, labels

train_ds = tfdf.keras.pd_dataframe_to_tf_dataset(preprocess_train_df,label="Survived").map(tokenize_names)
test_ds = tfdf.keras.pd_dataframe_to_tf_dataset(preprocess_test_df).map(tokenize_names)

In [55]:
model = tfdf.keras.GradientBoostedTreesModel(
    verbose=1, 
    features=[tfdf.keras.FeatureUsage(name=n) for n in input_features],
    exclude_non_specified_features=True,
    min_examples=1,
    categorical_algorithm="RANDOM",
    shrinkage=0.05,
    split_axis="SPARSE_OBLIQUE",
    sparse_oblique_normalization="MIN_MAX",
    sparse_oblique_num_projections_exponent=2.0,
    num_trees=2000,
    random_seed=1234,
    
)
model.fit(train_ds)

self_evaluation = model.make_inspector().evaluation()
print(f"Accuracy: {self_evaluation.accuracy} Loss:{self_evaluation.loss}")

Use /var/folders/3z/c5bxtqgx0yj70y6vbqnfwxtw0000gn/T/tmpgz9aqwy8 as temporary training directory
Reading training dataset...
Training dataset read in 0:00:00.097715. Found 891 examples.
Training model...


[WARNING 26-10-01 21:12:12.5594 +06 gradient_boosted_trees.cc:1886] "goss_alpha" set but "sampling_method" not equal to "GOSS".
[WARNING 26-10-01 21:12:12.5595 +06 gradient_boosted_trees.cc:1897] "goss_beta" set but "sampling_method" not equal to "GOSS".
[WARNING 26-10-01 21:12:12.5595 +06 gradient_boosted_trees.cc:1911] "selective_gradient_boosting_ratio" set but "sampling_method" not equal to "SELGB".


Model trained in 0:00:00.179060
Compiling model...
Model compiled.
Accuracy: 0.79347825050354 Loss:1.0656445026397705


[INFO 26-10-01 21:12:12.8410 +06 kernel.cc:1233] Loading model from path /var/folders/3z/c5bxtqgx0yj70y6vbqnfwxtw0000gn/T/tmpgz9aqwy8/model/ with prefix c8fa76c76206481c
[INFO 26-10-01 21:12:12.8428 +06 decision_forest.cc:660] Model loaded with 34 root(s), 1818 node(s), and 11 input feature(s).
[INFO 26-10-01 21:12:12.8428 +06 abstract_model.cc:1344] Engine "GradientBoostedTreesGeneric" built
[INFO 26-10-01 21:12:12.8428 +06 kernel.cc:1061] Use fast generic engine


In [56]:
model.summary()

Model: "gradient_boosted_trees_model_311"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
Total params: 1 (1.00 Byte)
Trainable params: 0 (0.00 Byte)
Non-trainable params: 1 (1.00 Byte)
_________________________________________________________________
Type: "GRADIENT_BOOSTED_TREES"
Task: CLASSIFICATION
Label: "__LABEL"

Input Features (11):
	Age
	Cabin
	Embarked
	Fare
	Name
	Parch
	Pclass
	Sex
	SibSp
	Ticket_item
	Ticket_number

No weights

Variable Importance: INV_MEAN_MIN_DEPTH:
    1.           "Sex"  0.588943 ################
    2.           "Age"  0.346745 ######
    3.          "Fare"  0.269876 ###
    4.          "Name"  0.205647 #
    5.         "Cabin"  0.185368 
    6.        "Pclass"  0.181872 
    7. "Ticket_number"  0.180223 
    8.   "Ticket_item"  0.175626 
    9.         "Parch"  0.174643 
   10.      "Embarked"  0.173688 
   11.         "SibSp"  0.171216 

Variable Importance: NUM_AS_R

In [ ]:

def prediction(model, test_df, passenger_ids, threshold=0.5):    
    proba_survive = model.predict(test_ds, verbose=0)[:, 0]
    
    return pd.DataFrame({
        "PassengerId": passenger_ids,
        "Survived": (proba_survive >= threshold).astype(int)
    })

k_prediction = prediction(model, test_ds, test_df["PassengerId"])

def make_submission(k_prediction):
    path = "/Users/aleksandr/Документы/TitanicML/data/gender_submission.csv"
    k_prediction.to_csv(path, index=False)
    print(f"Submission exported to {path}")

make_submission(k_prediction)


Submission exported to /Users/aleksandr/Документы/TitanicML/data/gender_submission.csv


In [ ]:
tuner = tfdf.tuner.RandomSearch(num_trials=1000)
tuner.choice("min_examples", [2, 5, 7, 10])
tuner.choice("categorical_algorithm", ["CART", "RANDOM"])

local_search_space = tuner.choice("growing_strategy", ["LOCAL"])
local_search_space.choice("max_depth", [3, 4, 5, 6, 8])

global_search_space = tuner.choice("growing_strategy", ["BEST_FIRST_GLOBAL"], merge=True)
global_search_space.choice("max_num_nodes", [16, 32, 64, 128, 256])

tuner.choice("shrinkage", [0.02, 0.05, 0.10, 0.15])
tuner.choice("num_candidate_attributes_ratio", [0.2, 0.5, 0.9, 1.0])


tuner.choice("split_axis", ["AXIS_ALIGNED"])
oblique_space = tuner.choice("split_axis", ["SPARSE_OBLIQUE"], merge=True)
oblique_space.choice("sparse_oblique_normalization",
                     ["NONE", "STANDARD_DEVIATION", "MIN_MAX"])
oblique_space.choice("sparse_oblique_weights", ["BINARY", "CONTINUOUS"])
oblique_space.choice("sparse_oblique_num_projections_exponent", [1.0, 1.5])

tuned_model = tfdf.keras.GradientBoostedTreesModel(tuner=tuner)
tuned_model.fit(train_ds, verbose=0)

tuned_self_evaluation = tuned_model.make_inspector().evaluation()
print(f"Accuracy: {tuned_self_evaluation.accuracy} Loss:{tuned_self_evaluation.loss}")

Use /var/folders/3z/c5bxtqgx0yj70y6vbqnfwxtw0000gn/T/tmpms6sr_at as temporary training directory


[WARNING 26-10-01 21:12:12.9665 +06 gradient_boosted_trees.cc:1886] "goss_alpha" set but "sampling_method" not equal to "GOSS".
[WARNING 26-10-01 21:12:12.9665 +06 gradient_boosted_trees.cc:1897] "goss_beta" set but "sampling_method" not equal to "GOSS".
[WARNING 26-10-01 21:12:12.9665 +06 gradient_boosted_trees.cc:1911] "selective_gradient_boosting_ratio" set but "sampling_method" not equal to "SELGB".


Accuracy: 0.8493150472640991 Loss:0.6777517199516296


[INFO 26-10-01 21:12:29.8705 +06 kernel.cc:1233] Loading model from path /var/folders/3z/c5bxtqgx0yj70y6vbqnfwxtw0000gn/T/tmpms6sr_at/model/ with prefix ef25c5d279e14c52
[INFO 26-10-01 21:12:29.8845 +06 decision_forest.cc:660] Model loaded with 84 root(s), 10236 node(s), and 13 input feature(s).
[INFO 26-10-01 21:12:29.8845 +06 abstract_model.cc:1344] Engine "GradientBoostedTreesGeneric" built
[INFO 26-10-01 21:12:29.8845 +06 kernel.cc:1061] Use fast generic engine


In [ ]:
predictions = None
num_predictions = 0

for i in range(100):
    print(f"i:{i}")
    model = tfdf.keras.GradientBoostedTreesModel(
        verbose=0, # Very few logs
        features=[tfdf.keras.FeatureUsage(name=n) for n in input_features],
        exclude_non_specified_features=True, 
        random_seed=i,
        honest=True,
    )
    model.fit(train_ds)
    
    sub_predictions = model.predict(test_ds, verbose=0)[:,0]
    if predictions is None:
        predictions = sub_predictions
    else:
        predictions += sub_predictions
    num_predictions += 1

predictions/=num_predictions

kaggle_predictions = pd.DataFrame({
        "PassengerId": test_df["PassengerId"],
        "Survived": (predictions >= 0.5).astype(int)
    })

make_submission(kaggle_predictions)

[WARNING 26-10-01 21:12:29.9604 +06 gradient_boosted_trees.cc:1886] "goss_alpha" set but "sampling_method" not equal to "GOSS".
[WARNING 26-10-01 21:12:29.9604 +06 gradient_boosted_trees.cc:1897] "goss_beta" set but "sampling_method" not equal to "GOSS".
[WARNING 26-10-01 21:12:29.9604 +06 gradient_boosted_trees.cc:1911] "selective_gradient_boosting_ratio" set but "sampling_method" not equal to "SELGB".


i:0


[INFO 26-10-01 21:12:30.1338 +06 kernel.cc:1233] Loading model from path /var/folders/3z/c5bxtqgx0yj70y6vbqnfwxtw0000gn/T/tmpi6b4xk5m/model/ with prefix 67ff310e140649d1
[INFO 26-10-01 21:12:30.1351 +06 kernel.cc:1061] Use fast generic engine
[WARNING 26-10-01 21:12:30.2242 +06 gradient_boosted_trees.cc:1886] "goss_alpha" set but "sampling_method" not equal to "GOSS".
[WARNING 26-10-01 21:12:30.2242 +06 gradient_boosted_trees.cc:1897] "goss_beta" set but "sampling_method" not equal to "GOSS".
[WARNING 26-10-01 21:12:30.2242 +06 gradient_boosted_trees.cc:1911] "selective_gradient_boosting_ratio" set but "sampling_method" not equal to "SELGB".


i:1


[INFO 26-10-01 21:12:30.4005 +06 kernel.cc:1233] Loading model from path /var/folders/3z/c5bxtqgx0yj70y6vbqnfwxtw0000gn/T/tmpz8437_yn/model/ with prefix 558c82f3f357417b
[INFO 26-10-01 21:12:30.4020 +06 kernel.cc:1061] Use fast generic engine
[WARNING 26-10-01 21:12:30.4922 +06 gradient_boosted_trees.cc:1886] "goss_alpha" set but "sampling_method" not equal to "GOSS".
[WARNING 26-10-01 21:12:30.4923 +06 gradient_boosted_trees.cc:1897] "goss_beta" set but "sampling_method" not equal to "GOSS".
[WARNING 26-10-01 21:12:30.4923 +06 gradient_boosted_trees.cc:1911] "selective_gradient_boosting_ratio" set but "sampling_method" not equal to "SELGB".


...

[INFO 26-10-01 21:13:00.2465 +06 kernel.cc:1233] Loading model from path /var/folders/3z/c5bxtqgx0yj70y6vbqnfwxtw0000gn/T/tmpobvrlp2a/model/ with prefix d8b287cabbc94a81
[INFO 26-10-01 21:13:00.2486 +06 abstract_model.cc:1344] Engine "GradientBoostedTreesQuickScorerExtended" built
[INFO 26-10-01 21:13:00.2487 +06 kernel.cc:1061] Use fast generic engine
[WARNING 26-10-01 21:13:00.3327 +06 gradient_boosted_trees.cc:1886] "goss_alpha" set but "sampling_method" not equal to "GOSS".
[WARNING 26-10-01 21:13:00.3327 +06 gradient_boosted_trees.cc:1897] "goss_beta" set but "sampling_method" not equal to "GOSS".
[WARNING 26-10-01 21:13:00.3328 +06 gradient_boosted_trees.cc:1911] "selective_gradient_boosting_ratio" set but "sampling_method" not equal to "SELGB".


i:98


[INFO 26-10-01 21:13:00.5563 +06 kernel.cc:1233] Loading model from path /var/folders/3z/c5bxtqgx0yj70y6vbqnfwxtw0000gn/T/tmpxxvrwzzh/model/ with prefix 263d0e999beb4424
[INFO 26-10-01 21:13:00.5587 +06 kernel.cc:1061] Use fast generic engine
[WARNING 26-10-01 21:13:00.6433 +06 gradient_boosted_trees.cc:1886] "goss_alpha" set but "sampling_method" not equal to "GOSS".
[WARNING 26-10-01 21:13:00.6434 +06 gradient_boosted_trees.cc:1897] "goss_beta" set but "sampling_method" not equal to "GOSS".
[WARNING 26-10-01 21:13:00.6434 +06 gradient_boosted_trees.cc:1911] "selective_gradient_boosting_ratio" set but "sampling_method" not equal to "SELGB".


i:99
Submission exported to /Users/aleksandr/Документы/TitanicML/data/gender_submission.csv


[INFO 26-10-01 21:13:00.8784 +06 kernel.cc:1233] Loading model from path /var/folders/3z/c5bxtqgx0yj70y6vbqnfwxtw0000gn/T/tmpimviyq93/model/ with prefix 28ae5ed6aa6c4cb6
[INFO 26-10-01 21:13:00.8811 +06 kernel.cc:1061] Use fast generic engine
